# DEAI-opdrachten Classificatie – Ames Housing

In deze verbeterde versie is de classificatiecode netter opgebouwd en sterker gemaakt.
De grootste verbeteringen zijn:

- consistente preprocessing met een `Pipeline`
- automatische behandeling van numerieke en categorische kolommen
- gebruik van `class_weight` bij scheve klassedistributies
- sterkere modellen met `RandomForestClassifier`
- extra tuning met `GridSearchCV` op basis van **macro F1-score**

In [1]:
import warnings
warnings.filterwarnings("ignore")

import logging
import pandas as pd
import numpy as np

from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

logger = logging.getLogger("classification_notebook")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))
    logger.addHandler(handler)

logger.propagate = False
logger.info("Classificatie-notebook is gestart")

INFO - Classificatie-notebook is gestart


In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

bestand = "AmesHousing.xlsx"

df = pd.read_excel(bestand, sheet_name="AmesHousing")
data_dictionary = pd.read_excel(bestand, sheet_name="Data Dictionary")
# Leest de dataset en de data dictionary uit het Excel-bestand in

logger.info(f"Dataset ingeladen met shape: {df.shape}")

display(df.head())
display(data_dictionary.head(20))

INFO - Dataset ingeladen met shape: (2930, 12)


,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story


,Variabele,Betekenis
0,ID,"Uniek nummer per huis, te vergelijken met een ..."
1,SalePrice,Verkoopprijs van het huis (in dollars: USD)
2,Garage,Geeft weer of het huis wel/geen garage bevat
3,Overall Qual,Algemene kwaliteit van materialen en afwerking...
4,Gr Liv Area,Woonoppervlak boven de grond (square feet)
5,Total Bsmt SF,Totale oppervlakte van de kelder
6,Lot Area,Grootte van het perceel (square feet)
7,Year Built,Bouwjaar van het huis
8,Full Bath,Aantal volledige badkamers
9,Bedroom AbvGr,Aantal slaapkamers boven de grond


In [3]:
logger.info("Verdeling van de targetklassen")

display(
    pd.DataFrame({
        "Garage count": df["Garage"].value_counts(dropna=False),
        "Garage aandeel": df["Garage"].value_counts(dropna=False, normalize=True).round(3)
    })
)

display(
    pd.DataFrame({
        "Overall Qual count": df["Overall Qual"].value_counts(dropna=False).sort_index(),
        "Overall Qual aandeel": df["Overall Qual"].value_counts(dropna=False, normalize=True).sort_index().round(3)
    })
)
# Laat zien dat vooral Garage een scheve verdeling heeft

INFO - Verdeling van de targetklassen


,Garage count,Garage aandeel
Garage,,
yes,2772,0.946
no,158,0.054


,Overall Qual count,Overall Qual aandeel
Overall Qual,,
1,4,0.001
2,13,0.004
3,40,0.014
4,226,0.077
5,825,0.282
6,732,0.250
7,602,0.205
8,350,0.119
9,107,0.037


In [4]:
def maak_preprocessor(dataframe, feature_kolommen):
    # Maakt een preprocessor voor numerieke en categorische kolommen
    X = dataframe[feature_kolommen].copy()

    categorische_kolommen = X.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()

    numerieke_kolommen = [kolom for kolom in X.columns if kolom not in categorische_kolommen]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                numerieke_kolommen
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorische_kolommen
            )
        ],
        remainder="drop"
    )

    return preprocessor, numerieke_kolommen, categorische_kolommen


def bereken_metrics(y_true, y_pred):
    # Berekent de belangrijkste classificatiescores
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }


def run_experiment(dataframe, target_kolom, feature_kolommen, model_klasse, model_kwargs, run_naam):
    # Voert één classificatie-experiment uit en geeft de resultaten terug
    data = dataframe[feature_kolommen + [target_kolom]].copy()
    data = data.dropna(subset=[target_kolom])

    X = data[feature_kolommen].copy()
    y = data[target_kolom].copy()

    preprocessor, numerieke_kolommen, categorische_kolommen = maak_preprocessor(data, feature_kolommen)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    logger.info(
        f"{run_naam}: train-test-split uitgevoerd | "
        f"X_train={X_train.shape}, X_test={X_test.shape}"
    )

    model = model_klasse(random_state=42, **model_kwargs)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = bereken_metrics(y_test, y_pred)
    labels = sorted(pd.Series(y).dropna().unique().tolist())
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    logger.info(
        f"{run_naam}: accuracy={metrics['accuracy']:.3f}, "
        f"balanced_accuracy={metrics['balanced_accuracy']:.3f}, "
        f"f1_macro={metrics['f1_macro']:.3f}"
    )

    return {
        "Run": run_naam,
        "Model": model_klasse.__name__,
        "features": feature_kolommen,
        "hyperparameters": model_kwargs,
        "numerieke_kolommen": numerieke_kolommen,
        "categorische_kolommen": categorische_kolommen,
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "precision_macro": metrics["precision_macro"],
        "recall_macro": metrics["recall_macro"],
        "f1_macro": metrics["f1_macro"],
        "confusion_matrix": cm,
        "labels": labels,
        "report": classification_report(y_test, y_pred, zero_division=0),
        "pipeline": pipeline,
        "X_train_shape": X_train.shape,
        "X_test_shape": X_test.shape,
        "y_train_shape": y_train.shape,
        "y_test_shape": y_test.shape
    }


def run_gridsearch_experiment(
    dataframe,
    target_kolom,
    feature_kolommen,
    model_klasse,
    base_model_kwargs,
    param_grid,
    run_naam,
    scoring="f1_macro"
):
    # Voert een GridSearchCV-experiment uit en gebruikt macro F1-score als hoofdmetric
    data = dataframe[feature_kolommen + [target_kolom]].copy()
    data = data.dropna(subset=[target_kolom])

    X = data[feature_kolommen].copy()
    y = data[target_kolom].copy()

    preprocessor, numerieke_kolommen, categorische_kolommen = maak_preprocessor(data, feature_kolommen)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    logger.info(
        f"{run_naam}: grid search gestart | "
        f"X_train={X_train.shape}, X_test={X_test.shape}"
    )

    model = model_klasse(random_state=42, **base_model_kwargs)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    zoek = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        refit=True
    )

    zoek.fit(X_train, y_train)

    best_model = zoek.best_estimator_
    y_pred = best_model.predict(X_test)
    metrics = bereken_metrics(y_test, y_pred)
    labels = sorted(pd.Series(y).dropna().unique().tolist())
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    best_params = {
        sleutel.replace("classifier__", ""): waarde
        for sleutel, waarde in zoek.best_params_.items()
    }

    logger.info(
        f"{run_naam}: beste cv-score={zoek.best_score_:.3f}, "
        f"accuracy={metrics['accuracy']:.3f}, "
        f"balanced_accuracy={metrics['balanced_accuracy']:.3f}, "
        f"f1_macro={metrics['f1_macro']:.3f}"
    )

    return {
        "Run": run_naam,
        "Model": model_klasse.__name__,
        "features": feature_kolommen,
        "hyperparameters": best_params,
        "best_cv_score": zoek.best_score_,
        "numerieke_kolommen": numerieke_kolommen,
        "categorische_kolommen": categorische_kolommen,
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "precision_macro": metrics["precision_macro"],
        "recall_macro": metrics["recall_macro"],
        "f1_macro": metrics["f1_macro"],
        "confusion_matrix": cm,
        "labels": labels,
        "report": classification_report(y_test, y_pred, zero_division=0),
        "pipeline": best_model,
        "X_train_shape": X_train.shape,
        "X_test_shape": X_test.shape,
        "y_train_shape": y_train.shape,
        "y_test_shape": y_test.shape
    }


def maak_resultaten_tabel(resultaten_dict):
    # Zet alle experimenten om in één overzichtelijke tabel
    rijen = []

    for naam, res in resultaten_dict.items():
        rij = {
            "Run": naam,
            "Model": res["Model"],
            "Features": ", ".join(res["features"]),
            "Hyperparameters": str(res["hyperparameters"]),
            "Accuracy": round(res["accuracy"], 4),
            "Balanced accuracy": round(res["balanced_accuracy"], 4),
            "Precision macro": round(res["precision_macro"], 4),
            "Recall macro": round(res["recall_macro"], 4),
            "F1 macro": round(res["f1_macro"], 4)
        }

        if "best_cv_score" in res:
            rij["Beste CV-score"] = round(res["best_cv_score"], 4)

        rijen.append(rij)

    resultaten_tabel = pd.DataFrame(rijen).sort_values(
        by=["F1 macro", "Balanced accuracy", "Accuracy"],
        ascending=False
    ).reset_index(drop=True)

    logger.info(f"Resultatentabel aangemaakt met {len(resultaten_tabel)} runs")
    return resultaten_tabel


def toon_confusion_matrix(resultaat):
    # Zet de confusion matrix van een run om in een tabel
    return pd.DataFrame(
        resultaat["confusion_matrix"],
        index=[f"Werkelijk: {label}" for label in resultaat["labels"]],
        columns=[f"Voorspeld: {label}" for label in resultaat["labels"]]
    )

## Model 1 – `Garage` voorspellen

In [5]:
garage_target = "Garage"

garage_features_basis = ["SalePrice", "Gr Liv Area", "Neighborhood"]
garage_features_rijk = [
    "SalePrice",
    "Overall Qual",
    "Gr Liv Area",
    "Total Bsmt SF",
    "Lot Area",
    "Year Built",
    "Full Bath",
    "Bedroom AbvGr",
    "Neighborhood",
    "House Style"
]
# Een rijkere featureset geeft het model meer informatie dan alleen de oorspronkelijke 3 features

garage_data = df[garage_features_rijk + [garage_target]].dropna(subset=[garage_target]).copy()

logger.info(f"Garage-data voorbereid met shape: {garage_data.shape}")
display(garage_data.head())

INFO - Garage-data voorbereid met shape: (2930, 11)


,SalePrice,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style,Garage
0,215000,6,1656,1080.0,31770,1960,1,3,NAmes,1Story,yes
1,105000,5,896,882.0,11622,1961,1,2,NAmes,1Story,yes
2,172000,6,1329,1329.0,14267,1958,1,3,NAmes,1Story,yes
3,244000,7,2110,2110.0,11160,1968,2,3,NAmes,1Story,yes
4,189900,5,1629,928.0,13830,1997,2,3,Gilbert,2Story,yes


In [6]:
garage_resultaten = {}

garage_experimenten = {
    "Decision Tree baseline": {
        "features": garage_features_basis,
        "model_klasse": DecisionTreeClassifier,
        "params": {
            "max_depth": 4,
            "min_samples_split": 10,
            "class_weight": "balanced"
        }
    },
    "Decision Tree rijkere features": {
        "features": garage_features_rijk,
        "model_klasse": DecisionTreeClassifier,
        "params": {
            "max_depth": 8,
            "min_samples_split": 12,
            "min_samples_leaf": 2,
            "class_weight": "balanced"
        }
    },
    "Random Forest rijkere features": {
        "features": garage_features_rijk,
        "model_klasse": RandomForestClassifier,
        "params": {
            "n_estimators": 300,
            "max_depth": 12,
            "min_samples_split": 6,
            "min_samples_leaf": 2,
            "class_weight": "balanced_subsample",
            "n_jobs": -1
        }
    }
}

for naam, info in garage_experimenten.items():
    garage_resultaten[naam] = run_experiment(
        dataframe=df,
        target_kolom=garage_target,
        feature_kolommen=info["features"],
        model_klasse=info["model_klasse"],
        model_kwargs=info["params"],
        run_naam=naam
    )

garage_tabel = maak_resultaten_tabel(garage_resultaten)
display(garage_tabel)

INFO - Decision Tree baseline: train-test-split uitgevoerd | X_train=(2344, 3), X_test=(586, 3)
INFO - Decision Tree baseline: accuracy=0.717, balanced_accuracy=0.777, f1_macro=0.536
INFO - Decision Tree rijkere features: train-test-split uitgevoerd | X_train=(2344, 10), X_test=(586, 10)
INFO - Decision Tree rijkere features: accuracy=0.843, balanced_accuracy=0.755, f1_macro=0.612
INFO - Random Forest rijkere features: train-test-split uitgevoerd | X_train=(2344, 10), X_test=(586, 10)
INFO - Random Forest rijkere features: accuracy=0.947, balanced_accuracy=0.736, f1_macro=0.740
INFO - Resultatentabel aangemaakt met 3 runs


,Run,Model,Features,Hyperparameters,Accuracy,Balanced accuracy,Precision macro,Recall macro,F1 macro
0,Random Forest rijkere features,RandomForestClassifier,"SalePrice, Overall Qual, Gr Liv Area, Total Bs...","{'n_estimators': 300, 'max_depth': 12, 'min_sa...",0.9471,0.7365,0.7437,0.7365,0.7400
1,Decision Tree rijkere features,DecisionTreeClassifier,"SalePrice, Overall Qual, Gr Liv Area, Total Bs...","{'max_depth': 8, 'min_samples_split': 12, 'min...",0.8430,0.7550,0.5916,0.7550,0.6124
2,Decision Tree baseline,DecisionTreeClassifier,"SalePrice, Gr Liv Area, Neighborhood","{'max_depth': 4, 'min_samples_split': 10, 'cla...",0.7167,0.7766,0.5655,0.7766,0.5355


In [7]:
garage_grid_resultaat = run_gridsearch_experiment(
    dataframe=df,
    target_kolom=garage_target,
    feature_kolommen=garage_features_rijk,
    model_klasse=RandomForestClassifier,
    base_model_kwargs={"n_jobs": -1},
    param_grid={
        "classifier__n_estimators": [200, 400],
        "classifier__max_depth": [8, 12, None],
        "classifier__min_samples_split": [4, 8],
        "classifier__min_samples_leaf": [1, 2],
        "classifier__class_weight": ["balanced", "balanced_subsample"]
    },
    run_naam="Random Forest grid search",
    scoring="f1_macro"
)

garage_resultaten["Random Forest grid search"] = garage_grid_resultaat

garage_tabel = maak_resultaten_tabel(garage_resultaten)
display(garage_tabel)

INFO - Random Forest grid search: grid search gestart | X_train=(2344, 10), X_test=(586, 10)
INFO - Random Forest grid search: beste cv-score=0.689, accuracy=0.947, balanced_accuracy=0.707, f1_macro=0.723
INFO - Resultatentabel aangemaakt met 4 runs


,Run,Model,Features,Hyperparameters,Accuracy,Balanced accuracy,Precision macro,Recall macro,F1 macro,Beste CV-score
0,Random Forest rijkere features,RandomForestClassifier,"SalePrice, Overall Qual, Gr Liv Area, Total Bs...","{'n_estimators': 300, 'max_depth': 12, 'min_sa...",0.9471,0.7365,0.7437,0.7365,0.7400,NaN
1,Random Forest grid search,RandomForestClassifier,"SalePrice, Overall Qual, Gr Liv Area, Total Bs...","{'class_weight': 'balanced', 'max_depth': None...",0.9471,0.7070,0.7432,0.7070,0.7234,0.6895
2,Decision Tree rijkere features,DecisionTreeClassifier,"SalePrice, Overall Qual, Gr Liv Area, Total Bs...","{'max_depth': 8, 'min_samples_split': 12, 'min...",0.8430,0.7550,0.5916,0.7550,0.6124,NaN
3,Decision Tree baseline,DecisionTreeClassifier,"SalePrice, Gr Liv Area, Neighborhood","{'max_depth': 4, 'min_samples_split': 10, 'cla...",0.7167,0.7766,0.5655,0.7766,0.5355,NaN


In [8]:
beste_garage_run = garage_tabel.iloc[0]["Run"]
beste_garage = garage_resultaten[beste_garage_run]

logger.info(f"Beste garage-run op basis van F1 macro: {beste_garage_run}")
print(beste_garage["report"])

garage_cm_df = toon_confusion_matrix(beste_garage)
display(garage_cm_df)

INFO - Beste garage-run op basis van F1 macro: Random Forest rijkere features


              precision    recall  f1-score   support

          no       0.52      0.50      0.51        32
         yes       0.97      0.97      0.97       554

    accuracy                           0.95       586
   macro avg       0.74      0.74      0.74       586
weighted avg       0.95      0.95      0.95       586



,Voorspeld: no,Voorspeld: yes
Werkelijk: no,16,16
Werkelijk: yes,15,539


## Model 2 – `Overall Qual` voorspellen

In [9]:
qual_target = "Overall Qual"

qual_features_basis = ["SalePrice", "Year Built", "Neighborhood"]
qual_features_rijk = [
    "SalePrice",
    "Garage",
    "Gr Liv Area",
    "Total Bsmt SF",
    "Lot Area",
    "Year Built",
    "Full Bath",
    "Bedroom AbvGr",
    "Neighborhood",
    "House Style"
]
# Voor Overall Qual gebruiken we een bredere featureset, omdat dit een moeilijker multi-class probleem is

qual_data = df[qual_features_rijk + [qual_target]].dropna(subset=[qual_target]).copy()

logger.info(f"Overall Qual-data voorbereid met shape: {qual_data.shape}")
display(qual_data.head())

INFO - Overall Qual-data voorbereid met shape: (2930, 11)


,SalePrice,Garage,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style,Overall Qual
0,215000,yes,1656,1080.0,31770,1960,1,3,NAmes,1Story,6
1,105000,yes,896,882.0,11622,1961,1,2,NAmes,1Story,5
2,172000,yes,1329,1329.0,14267,1958,1,3,NAmes,1Story,6
3,244000,yes,2110,2110.0,11160,1968,2,3,NAmes,1Story,7
4,189900,yes,1629,928.0,13830,1997,2,3,Gilbert,2Story,5


In [10]:
qual_resultaten = {}

qual_experimenten = {
    "Decision Tree baseline": {
        "features": qual_features_basis,
        "model_klasse": DecisionTreeClassifier,
        "params": {
            "max_depth": 7,
            "min_samples_split": 20,
            "min_samples_leaf": 5,
            "class_weight": "balanced"
        }
    },
    "Decision Tree rijkere features": {
        "features": qual_features_rijk,
        "model_klasse": DecisionTreeClassifier,
        "params": {
            "max_depth": 10,
            "min_samples_split": 12,
            "min_samples_leaf": 3,
            "class_weight": "balanced"
        }
    },
    "Random Forest rijkere features": {
        "features": qual_features_rijk,
        "model_klasse": RandomForestClassifier,
        "params": {
            "n_estimators": 400,
            "max_depth": 16,
            "min_samples_split": 6,
            "min_samples_leaf": 2,
            "class_weight": "balanced_subsample",
            "n_jobs": -1
        }
    }
}

for naam, info in qual_experimenten.items():
    qual_resultaten[naam] = run_experiment(
        dataframe=df,
        target_kolom=qual_target,
        feature_kolommen=info["features"],
        model_klasse=info["model_klasse"],
        model_kwargs=info["params"],
        run_naam=naam
    )

qual_tabel = maak_resultaten_tabel(qual_resultaten)
display(qual_tabel)

INFO - Decision Tree baseline: train-test-split uitgevoerd | X_train=(2344, 3), X_test=(586, 3)
INFO - Decision Tree baseline: accuracy=0.428, balanced_accuracy=0.407, f1_macro=0.311
INFO - Decision Tree rijkere features: train-test-split uitgevoerd | X_train=(2344, 10), X_test=(586, 10)
INFO - Decision Tree rijkere features: accuracy=0.473, balanced_accuracy=0.372, f1_macro=0.319
INFO - Random Forest rijkere features: train-test-split uitgevoerd | X_train=(2344, 10), X_test=(586, 10)
INFO - Random Forest rijkere features: accuracy=0.580, balanced_accuracy=0.419, f1_macro=0.400
INFO - Resultatentabel aangemaakt met 3 runs


,Run,Model,Features,Hyperparameters,Accuracy,Balanced accuracy,Precision macro,Recall macro,F1 macro
0,Random Forest rijkere features,RandomForestClassifier,"SalePrice, Garage, Gr Liv Area, Total Bsmt SF,...","{'n_estimators': 400, 'max_depth': 16, 'min_sa...",0.5802,0.4191,0.3873,0.4191,0.3995
1,Decision Tree rijkere features,DecisionTreeClassifier,"SalePrice, Garage, Gr Liv Area, Total Bsmt SF,...","{'max_depth': 10, 'min_samples_split': 12, 'mi...",0.4727,0.3721,0.3081,0.3721,0.3194
2,Decision Tree baseline,DecisionTreeClassifier,"SalePrice, Year Built, Neighborhood","{'max_depth': 7, 'min_samples_split': 20, 'min...",0.4283,0.4066,0.3085,0.4066,0.3113


In [11]:
qual_grid_resultaat = run_gridsearch_experiment(
    dataframe=df,
    target_kolom=qual_target,
    feature_kolommen=qual_features_rijk,
    model_klasse=RandomForestClassifier,
    base_model_kwargs={"n_jobs": -1},
    param_grid={
        "classifier__n_estimators": [250, 400],
        "classifier__max_depth": [12, 18, None],
        "classifier__min_samples_split": [4, 8],
        "classifier__min_samples_leaf": [1, 2],
        "classifier__class_weight": ["balanced", "balanced_subsample"]
    },
    run_naam="Random Forest grid search",
    scoring="f1_macro"
)

qual_resultaten["Random Forest grid search"] = qual_grid_resultaat

qual_tabel = maak_resultaten_tabel(qual_resultaten)
display(qual_tabel)

INFO - Random Forest grid search: grid search gestart | X_train=(2344, 10), X_test=(586, 10)
INFO - Random Forest grid search: beste cv-score=0.523, accuracy=0.611, balanced_accuracy=0.411, f1_macro=0.411
INFO - Resultatentabel aangemaakt met 4 runs


,Run,Model,Features,Hyperparameters,Accuracy,Balanced accuracy,Precision macro,Recall macro,F1 macro,Beste CV-score
0,Random Forest grid search,RandomForestClassifier,"SalePrice, Garage, Gr Liv Area, Total Bsmt SF,...","{'class_weight': 'balanced_subsample', 'max_de...",0.6109,0.4110,0.4148,0.4110,0.4107,0.523
1,Random Forest rijkere features,RandomForestClassifier,"SalePrice, Garage, Gr Liv Area, Total Bsmt SF,...","{'n_estimators': 400, 'max_depth': 16, 'min_sa...",0.5802,0.4191,0.3873,0.4191,0.3995,NaN
2,Decision Tree rijkere features,DecisionTreeClassifier,"SalePrice, Garage, Gr Liv Area, Total Bsmt SF,...","{'max_depth': 10, 'min_samples_split': 12, 'mi...",0.4727,0.3721,0.3081,0.3721,0.3194,NaN
3,Decision Tree baseline,DecisionTreeClassifier,"SalePrice, Year Built, Neighborhood","{'max_depth': 7, 'min_samples_split': 20, 'min...",0.4283,0.4066,0.3085,0.4066,0.3113,NaN


In [12]:
beste_qual_run = qual_tabel.iloc[0]["Run"]
beste_qual = qual_resultaten[beste_qual_run]

logger.info(f"Beste quality-run op basis van F1 macro: {beste_qual_run}")
print(beste_qual["report"])

qual_cm_df = toon_confusion_matrix(beste_qual)
display(qual_cm_df)

INFO - Beste quality-run op basis van F1 macro: Random Forest grid search


              precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         3
           3       0.33      0.25      0.29         8
           4       0.46      0.40      0.43        45
           5       0.66      0.72      0.69       165
           6       0.56      0.56      0.56       146
           7       0.66      0.62      0.64       121
           8       0.70      0.70      0.70        70
           9       0.55      0.52      0.54        21
          10       0.22      0.33      0.27         6

    accuracy                           0.61       586
   macro avg       0.41      0.41      0.41       586
weighted avg       0.61      0.61      0.61       586



,Voorspeld: 1,Voorspeld: 2,Voorspeld: 3,Voorspeld: 4,Voorspeld: 5,Voorspeld: 6,Voorspeld: 7,Voorspeld: 8,Voorspeld: 9,Voorspeld: 10
Werkelijk: 1,0,1,0,0,0,0,0,0,0,0
Werkelijk: 2,0,0,2,1,0,0,0,0,0,0
Werkelijk: 3,0,1,2,3,1,1,0,0,0,0
Werkelijk: 4,0,0,2,18,20,5,0,0,0,0
Werkelijk: 5,0,0,0,12,119,30,4,0,0,0
Werkelijk: 6,0,0,0,4,37,82,22,1,0,0
Werkelijk: 7,0,0,0,1,3,26,75,16,0,0
Werkelijk: 8,0,0,0,0,0,2,12,49,6,1
Werkelijk: 9,0,0,0,0,0,0,1,3,11,6
Werkelijk: 10,0,0,0,0,0,0,0,1,3,2


## Korte rapportagehulp

Gebruik in je conclusie vooral deze lijn:

- **Garage:** kijk naar `F1 macro` en `Balanced accuracy`, omdat de klassen scheef verdeeld zijn.
- **Overall Qual:** gebruik vooral `F1 macro`, omdat dit een multi-class probleem met 10 kwaliteitsniveaus is.
- De beste run is steeds de bovenste regel van de resultaatentabel, omdat die op `F1 macro` gesorteerd is.

Een nette formulering voor je verslag is:

> De verbeterde modellen combineren consistente preprocessing met krachtigere ensemble-modellen.
> Hierdoor kan de macro F1-score stijgen ten opzichte van een losse Decision Tree, vooral bij scheve of multi-class klassedistributies.